# Distribution changes along a time axis

The `changeforest` detector looks for points on a table's time axis
where the joint distribution of the numeric measures changes: the
rows after the point are drawn differently from the rows before it,
in a way no single column has to show. A correlation that switches
on for a stretch and off again, at constant means and variances in
every column, is the standard example. The method is Londschien,
Bühlmann and Kovács (2023), *Random Forests for Change Point
Detection*, Journal of Machine Learning Research 24; the pass runs
the `changeforest` crate, which implements it.

The detector is opt-in: request it with
`detectors=["changeforest"]` at the `standard` or `deep` budget. A
default profile records it as skipped in coverage, naming it
opt-in. It needs a time axis — declared with `hints={"time": ...}`
or assembled by the frame model — and at least one numeric measure
column; rows missing the axis value or any measure value are set
aside, and the coverage entry states how many complete rows were
segmented. Its findings are `changes` findings of type
`distribution_change`.

The [temporal analysis](temporal_analysis.ipynb) chapter covers
the single-series passes on the same axis. The
[state unemployment case study](../case_studies/unemployment_panel.ipynb)
works this detector through on a fifty-year panel.

## How a split is scored and kept

The pass runs binary segmentation over the complete rows in axis
order. Within a segment, a candidate split divides the rows into
before and after. A random forest is trained to distinguish the
before rows from the after rows, and the candidate's score — its
*gain* — is the classifier's log-likelihood ratio: how much better
the forest's before/after probabilities explain the segment than
one pooled distribution does. The candidate with the largest gain
is the segment's proposed split.

A proposed split is kept only if it clears a permutation bar. The
segment's rows are shuffled out of axis order 199 times, the
largest gain is recomputed on each shuffled copy, and the p-value
is the share of shuffled maxima at or above the observed maximum,
counting the observed arrangement itself: with 199 permutations
the smallest attainable p-value is 0.005. A split is stated when
its p-value is at most 0.01. Kept splits divide the segment, and
each part is searched the same way, so the search proceeds from
the strongest change in a segment to weaker ones inside its parts.

Every random draw in the procedure — the forests' construction
and the permutation streams — derives from one seed, fixed at 260
in the pass's code and recorded in each finding's evidence. Two
profiles of the same table state the same splits; a regression
test pins this at both the Rust and Python levels.

## Example: a covariance change

Five measures, 600 daily rows. On rows 200 to 399 the measures
share a common component, so each pair has correlation 0.7; on the
rows before and after, they are independent. Every measure keeps
mean 0 and variance 1 throughout, so no level, trend, or variance
statement about any single measure describes what changes at rows
200 and 400. The fixture is the change-in-covariance setup of the
method's paper.

In [ ]:
import datetime as dt
import math
import random

import polars as pl

import onceover

rng = random.Random(260)
measures = {f"m{j}": [] for j in range(5)}
for r in range(600):
    rho = 0.7 if 200 <= r < 400 else 0.0
    common = rng.gauss(0.0, 1.0)
    for j in range(5):
        own = rng.gauss(0.0, 1.0)
        measures[f"m{j}"].append(
            math.sqrt(rho) * common + math.sqrt(1.0 - rho) * own
        )
dates = [dt.date(2024, 1, 1) + dt.timedelta(days=i) for i in range(600)]
df = pl.DataFrame({"date": dates, **measures})

report = onceover.profile(df, detectors=["changeforest"])
changes = report.select(type="distribution_change")
for f in changes:
    print(f["claim"])
entry = report.coverage["changeforest"]
print(entry["status"], "-", entry["detail"])
assert len(changes) == 2

Both planted splits are stated, at rows 200 and 400 — the dates
2024-07-19 and 2025-02-04 in the fixture. The two gains are not
computed on the same rows. The split at row 400 carries gain
76.98, scored on the segment from row 200 to the end of the frame:
inside that segment, the forest separates the correlated rows
200–399 from the independent rows after them. The split at row
200 carries gain 18.12, scored on the whole frame, whose last 200
rows are independent on both sides of the comparison and dilute
it. A gain compares the two sides of one split within one
segment; it is not an effect size in the measures' units, and
gains from different segments are not on a common scale.

## Read the finding

The claim names the measures — the first four, with a count of
the rest — the axis, and the split's period, then gives the gain
and the p-value. The evidence carries the measurements behind
the claim:

In [ ]:
f = next(f for f in changes
         if f["evidence"][0]["metrics"]["split_row"] == 400)
m = f["evidence"][0]["metrics"]
for key in ("breakpoint", "split_row", "segment_start", "segment_end",
            "gain", "p_value", "alpha", "seed", "n_measures",
            "significant_splits"):
    print(f"{key}: {m[key]}")
print(pl.DataFrame(m["segments"]))

`breakpoint` is the axis value at the split, and `split_row`
its position among the segmented rows; the new segment starts at
that row. `segment_start` and `segment_end` bound the segment
inside which this split was scored — the rows its gain compares.
`significant_splits` is the total number of splits that cleared
the bar in the run; the stated findings can be fewer, under the
cap described below. The `segments` payload is the run's final
segmentation — the segments between consecutive stated splits —
and is the same on every finding of the run.

The payload lets a reader check a covariance claim directly.
The correlation between `m0` and `m1` inside each stated segment:

In [ ]:
for s in m["segments"]:
    rows = df.filter(
        (pl.col("date") >= dt.date.fromisoformat(s["from"]))
        & (pl.col("date") <= dt.date.fromisoformat(s["to"]))
    )
    corr = rows.select(pl.corr("m0", "m1")).item()
    print(f"segment {s['segment']} ({s['from']} to {s['to']}): "
          f"corr(m0, m1) = {corr:.2f}")

The middle segment's correlation is 0.72, against −0.12 and
0.13 in the segments either side: the dependence the forest
detected is in the segment table, readable without the model.

Do not confuse this type with `distribution_shift`, the
[drift](drift.ipynb) detector's type. Drift fits a model on a
reference period and scores how later periods' rows route
through it, one target at a time. `distribution_change` segments
the measures jointly along the axis and states where the rows'
joint distribution changes.

## A change of shape in one series

The joint case needs at least two measures; a single measure is
segmented the same way, and what the forest can then detect is a
change in the series' distribution beyond its level and spread.
The fixture folds the second half of a normal series to its
absolute value and re-centres and re-scales it, so the second
half holds the first half's mean and variance with a different
shape: a hard lower edge and a right tail. The series runs to
1,200 rows; at 300 rows a side this contrast sits at the edge of
the method's power at the 0.01 bar (the measurements are in the
[benchmark record](../../benchmarks/2026-11-11-changeforest-detector.md)).

In [ ]:
rng = random.Random(261)
raw = [rng.gauss(0.0, 1.0) for _ in range(1200)]
folded = [abs(v) for v in raw]
mean = sum(folded) / len(folded)
var = sum((v - mean) ** 2 for v in folded) / len(folded)
series = raw[:600] + [(v - mean) / math.sqrt(var) for v in folded[600:]]
shape = pl.DataFrame({
    "date": [dt.date(2024, 1, 1) + dt.timedelta(days=i) for i in range(1200)],
    "y": series,
})

shape_report = onceover.profile(shape, detectors=["changeforest"])
shape_changes = shape_report.select(type="distribution_change")
for f in shape_changes:
    print(f["claim"])
splits = [f["evidence"][0]["metrics"]["split_row"] for f in shape_changes]
assert any(570 <= s <= 630 for s in splits)


def describe(name, values):
    mu = sum(values) / len(values)
    sd = math.sqrt(sum((v - mu) ** 2 for v in values) / len(values))
    skew = sum(((v - mu) / sd) ** 3 for v in values) / len(values)
    print(f"{name}: mean {mu:+.2f}, sd {sd:.2f}, skewness {skew:+.2f}")


describe("rows 0-599 (normal)", series[:600])
describe("rows 600-1199 (folded)", series[600:])

The split is stated at row 600, the planted position. The two
halves' summaries show what the forest had to work with: means
−0.02 and +0.09, standard deviations 0.93 and 1.03 — and
skewness −0.12 against +0.92. A variance comparison sees the
two standard deviations; the shape difference is in the third
number.

## The single-series passes on the same frames

The `breaks` pass fits two-segment models of one series at a
time — segment means for `level_shift`, lines for
`trend_change`, segment variances for `variance_change`. On the
covariance frame there is nothing for those models to fit:

In [ ]:
breaks_report = onceover.profile(df, detectors=["breaks"])
changes_seen = [f for f in breaks_report.findings
                if f["category"] == "changes"]
print("breaks findings in the changes category:", len(changes_seen))

repeat = onceover.profile(df, detectors=["changeforest"])
print("repeat profile states identical findings:",
      repeat.findings == report.findings)
assert repeat.findings == report.findings

The breaks pass states nothing on the folded series either:
its variance model compares the two halves' spreads, which the
fixture holds close. The two passes are complementary. Where a
change moves a level, a trend, or a variance, breaks states it
with the fitted quantities; where the change is in the joint
distribution or the shape, this pass states the split and its
gain, and the segment summaries are the reader's check.

## Data and work limits

The bounds below are stated in the coverage entry when they
bind, and the count of significant splits is stated whether or
not the findings cap binds.

- At most 10,000 complete rows are segmented at the standard
  budget, 20,000 at deep. Past the cap, a stride sample in axis
  order is segmented, and split positions refer to the sampled
  rows.
- At most 32 measures are segmented jointly. Past the cap the
  first 32 measure columns are used and coverage names the ones
  set aside, as the unemployment case study shows.
- No segment shorter than 30 rows is split, so a frame needs
  more than 60 complete rows before any split can be stated.
- The search evaluates at most 512 segments.
- At most 10 splits are stated as findings, the largest by
  gain. `significant_splits` in each finding's evidence and
  the coverage entry give the total that cleared the bar.

The findings cap is the bound that binds in practice. Measured
before it was set, the unbounded search stated 418 significant
splits on a 20,000-row sample of hourly traffic data: at that
sampling grain the joint distribution of weather and traffic
measures changes from hour to hour, and each change clears the
bar on enough rows. A report cannot carry hundreds of splits as
findings. The pass states the ten largest and discloses the
total, so a reader can tell a frame with one regime change from
a frame whose distribution changes throughout.

## Interpretation

- A stated split locates where the joint distribution changes.
  It does not say which aspect changed — dependence, shape, or
  a level the other passes also state — and it does not name a
  cause. Read the segment summaries and the breaks findings on
  the same frame before interpreting a split.
- The breakpoint is the first row of the new segment, stated
  as its axis value. The location is the maximum-gain position
  inside its segment; neighbouring positions with nearly the
  same gain are not reported, so read the date as the centre of
  a short interval, not as an exact day.
- The p-value is a permutation value within one segment, at a
  0.01 bar. It is not adjusted across the segments of a run,
  and it is not a probability that the split is real.
- `report.verify` does not recount these statements: the gain
  and the p-value are outputs of the fitted forests, with no
  exact recount from the data. The verifier marks each
  `distribution_change` statement `not_checkable`, as it does
  the breaks statements, and the finding's evidence says so in
  its `validation` entry.
- A change confined to a few rows, or to a stretch shorter
  than the 30-row minimum segment, is outside this pass; the
  breaks pass's excursion scan covers short departures in one
  series.

## Checks

The implementation is tested on the fixtures of this chapter —
the covariance setup, the folded shape change, a pure mean
shift as the breaks control, and no-change frames, which state
nothing — and on determinism: two profiles of the covariance
fixture produce identical findings. The acceptance
measurements, the power probes at shorter lengths, and the
wall times on corpus frames are recorded in
`docs/benchmarks/2026-11-11-changeforest-detector.md`.